In [1]:
!pip install -q transformers openpyxl

import json
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score
import openpyxl
import zipfile

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ĐƯỜNG DẪN DATASET MỚI
# ═══════════════════════════════════════════════════════════════════════════
DATASET_BASE = "/kaggle/input/datasets/thaiphonghuan/scoring/vietname-dataset-scoring - training"
CV_DIR = os.path.join(DATASET_BASE, "cv")
JD_DIR = os.path.join(DATASET_BASE, "jd")
SPLIT_ROOT = os.path.join(DATASET_BASE, "dataset")

CV_VIEWS = ['skills', 'education', 'year_experience', 'experience', 'responsibilities',
            'projects', 'publications', 'achievement', 'certification', 'employment_history']
JD_VIEWS = ['skills', 'education', 'year_experience_requirement', 'experience_requirement', 'responsibilities']

# === CROSS-ENCODER CONFIG ===
ENCODER_NAME = "keepitreal/vietnamese-sbert"
ENCODER_SLUG = "keepitreal-vietnamese-sbert/vietnamese-bi-encoder_vn_scoring"
FUSION_NAME = "weighted_sum"
CACHE_DIR = "/kaggle/working/cache"
CKPT_DIR = "/kaggle/working/checkpoints"
EXCEL_PATH = "/kaggle/working/results_crossencoder_vn.xlsx"
MAX_LEN = 256

#   "BAAI/bge-reranker-v2-m3"
#   "jinaai/jina-reranker-v2-base-multilingual"
#   "xlm-roberta-base"
#   "bert-base-multilingual-cased"

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
encoder = AutoModel.from_pretrained(ENCODER_NAME).to(device).eval()
for p in encoder.parameters():
    p.requires_grad_(False)
HIDDEN_DIM = encoder.config.hidden_size

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm

from transformers.utils import logging
logging.set_verbosity_error()

import warnings
warnings.filterwarnings("ignore")

def process_view(view_data):
    """Xử lý missing_section thành special token [EMPTY_SECTION]; trả về (text, exists_flag)."""
    if view_data is None: return "[EMPTY_SECTION]", 0
    if isinstance(view_data, list):
        clean_list = [str(x) for x in view_data if str(x).lower() != "missing_section"]
        if not clean_list:
            return "[EMPTY_SECTION]", 0
        return ". ".join(clean_list), 1
    val = str(view_data).lower()
    if val in ("missing_section", "", "none"):
        return "[EMPTY_SECTION]", 0
    return str(view_data), 1

def prepare_data_from_csv(split_name):
    """Đọc CSV và trích xuất CLS vector (768-dim)."""

    # ✅ FIX 1: Xử lý ENCODER_SLUG để không tạo ra đường dẫn lỗi (thay / bằng _)
    safe_slug = ENCODER_SLUG.replace('/', '_')
    # ⚠️ Bump version cache ('_v2fix') để KHÔNG nạp lại cache hỏng từ lần chạy lỗi parsing trước.
    cache_path = f"{CACHE_DIR}/{safe_slug}_{split_name}_scoring_v2fix.npz"
    
    # ✅ FIX 2: Đảm bảo thư mục cache tồn tại
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)

    if os.path.exists(cache_path):
        d = np.load(cache_path)
        print(f"📦 Loaded cache: {cache_path} | cls.shape={d['cls'].shape}")
        return d['cls'], d['mask'], d['y_reg'], d['y_bin']

    # Xác định đường dẫn CSV
    match_csv = os.path.join(SPLIT_ROOT, split_name, f"match/{split_name}_match.csv")
    mismatch_csv = os.path.join(SPLIT_ROOT, split_name, f"mismatch/{split_name}_mismatch.csv")

    configs = [(match_csv, 1), (mismatch_csv, 0)]
    all_cls, all_mask, all_yreg, all_ybin = [], [], [], []
    max_len = min(MAX_LEN, tokenizer.model_max_length)

    print(f"🚀 Processing split: {split_name} | Encoder: {ENCODER_NAME}")
    
    for csv_path, b_label in configs:
        if not os.path.exists(csv_path):
            print(f"⚠️ Không tìm thấy {csv_path}")
            continue

        df = pd.read_csv(csv_path)
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"CSV {b_label}"):
            # Load JSON file với encoding utf-8-sig để tránh lỗi BOM tiếng Việt
            # ✅ FIX (ROOT CAUSE): các file CV/JD lưu view ở TOP-LEVEL, không bọc trong
            # 'resume_structured'/'job_description_structured'. Dùng fallback về chính object
            # để không trả về {} (khiến mọi view rỗng -> mask toàn 0 -> model đoán hằng số).
            with open(os.path.join(CV_DIR, row['cv']), 'r', encoding='utf-8-sig') as f:
                cv_obj = json.load(f)
                cv_j = cv_obj.get('resume_structured', cv_obj)
            with open(os.path.join(JD_DIR, row['jd']), 'r', encoding='utf-8-sig') as f:
                jd_obj = json.load(f)
                jd_j = jd_obj.get('job_description_structured', jd_obj)

            # Trích xuất văn bản từng view
            cv_texts, cv_exists = zip(*[process_view(cv_j.get(v)) for v in CV_VIEWS])
            jd_texts, jd_exists = zip(*[process_view(jd_j.get(v)) for v in JD_VIEWS])

            # Tạo 50 cặp (Cartesian product)
            cv_batch = [cv_texts[i] for i in range(10) for j in range(5)]
            jd_batch = [jd_texts[j] for i in range(10) for j in range(5)]
            
            # Tokenize và Forward qua Encoder (Frozen)
            tok = tokenizer(cv_batch, jd_batch, padding=True, truncation=True, 
                            max_length=max_len, return_tensors='pt').to(device)
            
            with torch.no_grad():
                # Lấy vector CLS (token đầu tiên) của 50 cặp
                cls = encoder(**tok).last_hidden_state[:, 0, :].cpu().numpy()

            all_cls.append(cls)
            # Tạo mask 10x5 -> flatten thành 50
            all_mask.append(np.outer(np.array(cv_exists), np.array(jd_exists)).flatten().astype(np.float32))
            all_yreg.append(float(row['score']))
            all_ybin.append(b_label)

    # Chuyển thành numpy array
    X = np.stack(all_cls).astype(np.float32)
    M = np.stack(all_mask).astype(np.float32)
    Yr = np.array(all_yreg, dtype=np.float32)
    Yb = np.array(all_ybin, dtype=np.float32)

    # ✅ SANITY CHECK: nếu mask trung bình ~0 nghĩa là parsing JSON vẫn sai (input toàn rỗng)
    print(f"🔎 Sanity | mask active fraction = {M.mean():.3f} (phải > 0, KHÔNG được = 0.000)")

    # ✅ Lưu cache
    np.savez_compressed(cache_path, cls=X, mask=M, y_reg=Yr, y_bin=Yb)
    print(f"💾 Saved cache: {cache_path}")

    return X, M, Yr, Yb

# THỰC THI
X_train, M_train, y_train, b_train = prepare_data_from_csv("train")
X_val,   M_val,   y_val,   b_val   = prepare_data_from_csv("val")
X_test,  M_test,  y_test,  b_test  = prepare_data_from_csv("test")

print(f"✅ Dữ liệu hoàn tất: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

# jina

In [1]:
!pip install -q transformers openpyxl

import json
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, XLMRobertaModel
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score
import openpyxl
import zipfile

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, XLMRobertaModel
import os

# ═══════════════════════════════════════════════════════════════════════════
# 1. CẤU HÌNH ĐƯỜNG DẪN (Dataset Việt Nam)
# ═══════════════════════════════════════════════════════════════════════════
DATASET_ROOT = "/kaggle/input/datasets/thaiphonghuan/scoring/vietname-dataset-scoring - training"
CV_DIR = os.path.join(DATASET_ROOT, "cv")
JD_DIR = os.path.join(DATASET_ROOT, "jd")
SPLIT_ROOT = os.path.join(DATASET_ROOT, "dataset")

CV_VIEWS = ['skills', 'education', 'year_experience', 'experience', 'responsibilities',
            'projects', 'publications', 'achievement', 'certification', 'employment_history']
JD_VIEWS = ['skills', 'education', 'year_experience_requirement', 'experience_requirement', 'responsibilities']

# Cấu hình Jina
ENCODER_NAME = "jinaai/jina-reranker-v2-base-multilingual"
ENCODER_SLUG = "jina_reranker_v2_vn"
FUSION_NAME = "weighted_sum"
CACHE_DIR = "/kaggle/working/cache"
MAX_LEN = 256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device}")

# Load Jina thông qua XLMRobertaModel để bypass lỗi custom code
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
# Ép sang float32 để tương thích hoàn toàn với numpy cache
encoder = XLMRobertaModel.from_pretrained(ENCODER_NAME, torch_dtype=torch.float32).to(device).eval()

for p in encoder.parameters():
    p.requires_grad_(False)

HIDDEN_DIM = encoder.config.hidden_size
print(f"✅ Encoder Jina loaded: {ENCODER_NAME} | hidden_dim={HIDDEN_DIM}")

In [ ]:
import json
import pandas as pd
import numpy as np
from tqdm import tqdm

def process_view(view_data):
    if view_data is None: return "[EMPTY_SECTION]", 0
    if isinstance(view_data, list):
        clean_list = [str(x) for x in view_data if str(x).lower() != "missing_section"]
        if not clean_list: return "[EMPTY_SECTION]", 0
        return ". ".join(clean_list), 1
    val = str(view_data).lower()
    if val in ("missing_section", "", "none"): return "[EMPTY_SECTION]", 0
    return str(view_data), 1

def prepare_data_from_csv_jina(split_name):
    """Đọc CSV split tương ứng và trích xuất đặc trưng bằng Jina backbone."""
    
    # Xử lý path an toàn
    safe_slug = ENCODER_SLUG.replace('/', '_')
    cache_path = f"{CACHE_DIR}/{safe_slug}_{split_name}_scoring_v2fix.npz"
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)

    if os.path.exists(cache_path):
        d = np.load(cache_path)
        print(f"📦 Loaded Jina cache: {cache_path} | cls.shape={d['cls'].shape}")
        return d['cls'], d['mask'], d['y_reg'], d['y_bin']

    # Đường dẫn CSV
    match_csv = os.path.join(SPLIT_ROOT, split_name, f"match/{split_name}_match.csv")
    mismatch_csv = os.path.join(SPLIT_ROOT, split_name, f"mismatch/{split_name}_mismatch.csv")

    configs = [(match_csv, 1), (mismatch_csv, 0)]
    all_cls, all_mask, all_yreg, all_ybin = [], [], [], []
    max_len = min(MAX_LEN, tokenizer.model_max_length)

    print(f"🚀 Processing Jina encoding for: {split_name}")
    
    for csv_path, b_label in configs:
        if not os.path.exists(csv_path):
            print(f"⚠️ Không tìm thấy {csv_path}")
            continue

        df = pd.read_csv(csv_path)
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"CSV {b_label}"):
            # Load JSON với logic fallback (Root key check)
            with open(os.path.join(CV_DIR, row['cv']), 'r', encoding='utf-8-sig') as f:
                cv_obj = json.load(f)
                cv_j = cv_obj.get('resume_structured', cv_obj)
            with open(os.path.join(JD_DIR, row['jd']), 'r', encoding='utf-8-sig') as f:
                jd_obj = json.load(f)
                jd_j = jd_obj.get('job_description_structured', jd_obj)

            # Trích xuất views
            cv_texts, cv_exists = zip(*[process_view(cv_j.get(v)) for v in CV_VIEWS])
            jd_texts, jd_exists = zip(*[process_view(jd_j.get(v)) for v in JD_VIEWS])

            # Tạo batch 50 cặp view
            cv_batch = [cv_texts[i] for i in range(10) for j in range(5)]
            jd_batch = [jd_texts[j] for i in range(10) for j in range(5)]
            
            # Forward qua Jina Backbone
            tok = tokenizer(cv_batch, jd_batch, padding=True, truncation=True, 
                            max_length=max_len, return_tensors='pt').to(device)
            
            with torch.no_grad():
                # Lấy vector CLS của Jina
                cls = encoder(**tok).last_hidden_state[:, 0, :].cpu().numpy()

            all_cls.append(cls)
            all_mask.append(np.outer(np.array(cv_exists), np.array(jd_exists)).flatten().astype(np.float32))
            all_yreg.append(float(row['score']))
            all_ybin.append(b_label)

    X = np.stack(all_cls).astype(np.float32)
    M = np.stack(all_mask).astype(np.float32)
    Yr = np.array(all_yreg, dtype=np.float32)
    Yb = np.array(all_ybin, dtype=np.float32)

    # Sanity check
    print(f"🔎 Sanity | mask active fraction = {M.mean():.3f}")

    np.savez_compressed(cache_path, cls=X, mask=M, y_reg=Yr, y_bin=Yb)
    print(f"💾 Saved Jina cache: {cache_path}")
    return X, M, Yr, Yb

# THỰC THI ENCODING
X_train, M_train, y_train, b_train = prepare_data_from_csv_jina("train")
X_val,   M_val,   y_val,   b_val   = prepare_data_from_csv_jina("val")
X_test,  M_test,  y_test,  b_test  = prepare_data_from_csv_jina("test")

print(f"✅ Jina Encoding hoàn tất: Train={X_train.shape[0]}")

In [ ]:
import os

CACHE_DIR = "/kaggle/working/cache"
CKPT_DIR = "/kaggle/working/checkpoints"

EXCEL_PATH = "/kaggle/working/results_crossencoder_vn.xlsx"

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

FUSION_NAME = "weighted_sum" 
ENCODER_NAME = "jinaai/jina-reranker-v2-base-multilingual"
ENCODER_SLUG = "jina_reranker_v2_vn"

safe_slug = ENCODER_SLUG.replace('/', '_')

CKPT_PATH = f"{CKPT_DIR}/{FUSION_NAME}_{safe_slug}_baseline_sync.pth"

print(f"✅ Đã cấu hình xong hệ thống đường dẫn:")
print(f"   - File Excel kết quả: {EXCEL_PATH}")
print(f"   - File Model nạp vào: {CKPT_PATH}")
print(f"   - Tên mô hình trong báo cáo: {ENCODER_NAME}")

# Weighted sum

In [ ]:
class CrossEncDataset(Dataset):
    def __init__(self, cls, mask, y_reg, y_bin):
        self.cls = torch.tensor(cls, dtype=torch.float32)       # [N, 50, H]
        self.mask = torch.tensor(mask, dtype=torch.float32)     # [N, 50]
        self.y_reg = torch.tensor(y_reg, dtype=torch.float32).view(-1, 1)
        self.y_bin = torch.tensor(y_bin, dtype=torch.float32).view(-1, 1)
    def __len__(self): return len(self.cls)
    def __getitem__(self, idx):
        return self.cls[idx], self.mask[idx], self.y_reg[idx], self.y_bin[idx]

class ViewPairScorer(nn.Module):
    """Mã hóa tương tác: 2 tầng Linear, bias khởi tạo -2.0."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
        nn.init.constant_(self.fc[3].bias, -2.0) 

    def forward(self, cls):
        return torch.sigmoid(self.fc(cls)).squeeze(-1)

class WeightedSumFusion(nn.Module):
    """Bản tuyến tính tương đương: Nhận 100 features (50 scores + 50 masks)."""
    def __init__(self, num_views=50):
        super().__init__()
        self.weighted_layer = nn.Linear(num_views * 2, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, a_ij, mask):
        # Ghép cặp (concatenation) 100 chiều
        x = torch.cat([a_ij, mask], dim=-1) 
        return self.sigmoid(self.weighted_layer(x))

class FusionWithScorer(nn.Module):
    def __init__(self, hidden_dim, fusion_module):
        super().__init__()
        self.scorer = ViewPairScorer(hidden_dim)
        self.fusion = fusion_module
        
    def forward(self, cls, mask):
        raw_a_ij = self.scorer(cls) 
        masked_a_ij = raw_a_ij * mask 
        final_score = self.fusion(masked_a_ij, mask)               
        return final_score, masked_a_ij, raw_a_ij                                 

# Loaders
train_loader = DataLoader(CrossEncDataset(X_train, M_train, y_train, b_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(CrossEncDataset(X_val,   M_val,   y_val,   b_val),   batch_size=32)
test_loader  = DataLoader(CrossEncDataset(X_test,  M_test,  y_test,  b_test),  batch_size=32)

model = FusionWithScorer(HIDDEN_DIM, WeightedSumFusion()).to(device)

# --- HYPERPARAMETERS ĐỒNG BỘ TUYỆT ĐỐI ---
criterion_reg = nn.MSELoss() 
criterion_bin = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01) 
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

print(f"✅ WEIGHTED SUM READY | Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# =========================
# Training config
# =========================
epochs = 150
best_val_loss = float('inf')

safe_slug = ENCODER_SLUG.replace('/', '_')
CKPT_PATH = f"{CKPT_DIR}/{FUSION_NAME}_{safe_slug}.pth"
os.makedirs(CKPT_DIR, exist_ok=True)

# Hệ số phạt đồng bộ
LAMBDA_PENALTY = 2.0 

# History
train_losses = []
val_losses = []
lrs = []

print(f"🔥 Bắt đầu Training {FUSION_NAME} | Sync Mode (BCE weight: 0.5) | encoder={ENCODER_NAME}")

for epoch in range(epochs):
    # =========================
    # Train
    # =========================
    model.train()
    total_train_loss = 0.0
    
    for cls, mask, by_reg, by_bin in train_loader:
        cls, mask, by_reg, by_bin = cls.to(device), mask.to(device), by_reg.to(device), by_bin.to(device)
        optimizer.zero_grad()
        
        pred, masked_a_ij, raw_a_ij = model(cls, mask)
        
        task_loss = criterion_reg(pred, by_reg) + 0.5 * criterion_bin(pred, by_bin)
        
        empty_penalty = torch.mean(((1 - mask) * raw_a_ij) ** 2)
        
        activity_loss = torch.mean(mask * (0.1 - masked_a_ij).clamp(min=0)) 
        
        loss = task_loss + (LAMBDA_PENALTY * empty_penalty) + (0.5 * activity_loss)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()

    avg_train = total_train_loss / len(train_loader)

    # =========================
    # Validation
    # =========================
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for cls, mask, by_reg, by_bin in val_loader:
            cls, mask, by_reg, by_bin = cls.to(device), mask.to(device), by_reg.to(device), by_bin.to(device)
            out, m_aij, r_aij = model(cls, mask)
            
            # ✅ ĐỒNG BỘ: Tính Val Task Loss với hệ số 0.5
            v_task = criterion_reg(out, by_reg) + 0.5 * criterion_bin(out, by_bin)
            
            # ✅ ĐỒNG BỘ: Tính đầy đủ Penalty và Activity cho Val
            v_pen = torch.mean(((1 - mask) * r_aij) ** 2)
            v_act = torch.mean(mask * (0.1 - m_aij).clamp(min=0))
            
            # Tổng hợp Val Loss (Sử dụng đúng công thức như lúc Train)
            total_val_loss += (v_task + (LAMBDA_PENALTY * v_pen) + (0.5 * v_act)).item()

    avg_val = total_val_loss / len(val_loader)
    scheduler.step(avg_val)

    train_losses.append(avg_train)
    val_losses.append(avg_val)
    lrs.append(optimizer.param_groups[0]['lr'])

    if (epoch + 1) % 10 == 0:
        lr_cur = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:3d} | Train: {avg_train:.6f} | Val: {avg_val:.6f} | LR: {lr_cur:.6e}")

    # Checkpoint (Lưu model tốt nhất dựa trên Val Loss tổng hợp)
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
        torch.save({
            'scorer': model.scorer.state_dict(),
            'fusion': model.fusion.state_dict(),
            'hidden_dim': HIDDEN_DIM,
            'encoder': ENCODER_NAME,
            'epoch': epoch + 1,
            'val_loss': avg_val
        }, CKPT_PATH)

print(f"✅ Hoàn tất Training! Best Val Loss (inc. Penalty): {best_val_loss:.6f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Safety check
# =========================
assert 'train_losses' in globals(), "❌ Chưa có train_losses"
assert 'val_losses' in globals(), "❌ Chưa có val_losses"

# =========================
# Data
# =========================
epochs = len(train_losses)
x = np.arange(1, epochs + 1)

final_train = train_losses[-1]
final_val = val_losses[-1]

# =========================
# Plot
# =========================
plt.figure(figsize=(8, 5))
plt.plot(
    x,
    train_losses,
    label=f"Train Loss (final = {final_train:.4f})"
)
plt.plot(
    x,
    val_losses,
    label=f"Validation Loss (final = {final_val:.4f})"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()      # ✅ chỉ show
plt.close()

In [ ]:
import torch
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from tqdm import tqdm

# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD BEST CHECKPOINT
# ═══════════════════════════════════════════════════════════════════════════
print(f"📂 Loading best model from: {CKPT_PATH}")
checkpoint = torch.load(CKPT_PATH, map_location=device)

model.scorer.load_state_dict(checkpoint['scorer'])
model.fusion.load_state_dict(checkpoint['fusion'])
model.eval()

all_preds = []
all_true_reg = []
all_true_bin = []

# ═══════════════════════════════════════════════════════════════════════════
# 2. TESTING
# ═══════════════════════════════════════════════════════════════════════════
print("🚀 Đang thực hiện dự đoán trên tập Test (Vietnamese MTL)...")
with torch.no_grad():
    for cls, mask, by_reg, by_bin in tqdm(test_loader, desc="Testing"):
        cls, mask = cls.to(device), mask.to(device)

        outputs, _, _ = model(cls, mask)

        all_preds.extend(outputs.cpu().numpy().flatten())
        all_true_reg.extend(by_reg.numpy().flatten())
        all_true_bin.extend(by_bin.numpy().flatten())

all_preds = np.array(all_preds)
all_true_reg = np.array(all_true_reg)
all_true_bin = np.array(all_true_bin)

# Sanity check
print(
    f"🔎 Sanity | preds: mean={all_preds.mean():.4f} "
    f"std={all_preds.std():.4f} "
    f"min={all_preds.min():.4f} "
    f"max={all_preds.max():.4f}"
)

# Binary threshold
threshold = 0.5
all_preds_bin = (all_preds >= threshold).astype(int)

# ═══════════════════════════════════════════════════════════════════════════
# 3. METRICS
# ═══════════════════════════════════════════════════════════════════════════
mae = mean_absolute_error(all_true_reg, all_preds)
mse = mean_squared_error(all_true_reg, all_preds)

acc = accuracy_score(all_true_bin, all_preds_bin)
pre = precision_score(all_true_bin, all_preds_bin, zero_division=0)
rec = recall_score(all_true_bin, all_preds_bin, zero_division=0)
f1  = f1_score(all_true_bin, all_preds_bin, zero_division=0)

print("\n" + "="*60)
print("📊 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (VIETNAMESE)")
print("-" * 60)
print(f"Encoder: {ENCODER_NAME}")
print(f"Fusion : {FUSION_NAME}")
print("-" * 60)
print(f"{'MAE':<25} | {mae:.4f}")
print(f"{'MSE':<25} | {mse:.4f}")
print(f"{'Accuracy':<25} | {acc*100:.2f}%")
print(f"{'Precision':<25} | {pre:.4f}")
print(f"{'Recall':<25} | {rec:.4f}")
print(f"{'F1-Score':<25} | {f1:.4f}")
print("="*60)

# ═══════════════════════════════════════════════════════════════════════════
# 4. HIỂN THỊ ĐỐI CHIẾU 10 MẪU ĐẦU TIÊN
# ═══════════════════════════════════════════════════════════════════════════
print("\n🔍 Đối chiếu mẫu thực tế:")
print(f"{'ID':<4} | {'Thực tế':<10} | {'Điểm Gốc':<10} | {'AI Dự đoán':<10} | {'Kết luận'}")
print("-" * 65)
for i in range(min(10, len(all_preds))):
    label_true = "Match" if all_true_bin[i] == 1 else "Mismatch"
    label_pred = "Match" if all_preds_bin[i] == 1 else "Mismatch"
    status = "✅" if all_true_bin[i] == all_preds_bin[i] else "❌"
    print(f"{i:<4} | {label_true:<10} | {all_true_reg[i]:<10.4f} | {all_preds[i]:<10.4f} | {status} {label_pred}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import json
import os
import numpy as np
import torch

# ═══════════════════════════════════════════════════════════════════════════
# 1. CHUẨN BỊ DỮ LIỆU (Sử dụng 3 mẫu từ tập Test Match)
# ═══════════════════════════════════════════════════════════════════════════
test_match_df = pd.read_csv(os.path.join(SPLIT_ROOT, "test/match/test_match.csv"))
# Lấy 3 mẫu ngẫu nhiên để vẽ
sample_to_vis = test_match_df.sample(3, random_state=42).to_dict('records')

def get_clean_label(view_name, text, is_cv=True):
    if not text or text == "[EMPTY_SECTION]":
        return f"[{view_name[:4].upper()}]\n(Trống)"
    limit = 35
    short = (text[:limit] + '..') if len(text) > limit else text
    return f"[{view_name[:4].upper()}] {short}" if is_cv else f"[{view_name[:4].upper()}]\n{short}"

# ═══════════════════════════════════════════════════════════════════════════
# 2. HÀM TÍNH TOÁN ALIGNMENT CHI TIẾT
# ═══════════════════════════════════════════════════════════════════════════
def compute_heatmap_data(row):
    # Nạp JSON (Sử dụng đúng logic fallback của bạn)
    with open(os.path.join(CV_DIR, row['cv']), 'r', encoding='utf-8-sig') as f:
        cv_obj = json.load(f)
        cv_j = cv_obj.get('resume_structured', cv_obj)
    with open(os.path.join(JD_DIR, row['jd']), 'r', encoding='utf-8-sig') as f:
        jd_obj = json.load(f)
        jd_j = jd_obj.get('job_description_structured', jd_obj)

    # Trích xuất views
    cv_texts, cv_exists = zip(*[process_view(cv_j.get(v)) for v in CV_VIEWS])
    jd_texts, jd_exists = zip(*[process_view(jd_j.get(v)) for v in JD_VIEWS])
    m_ij = np.outer(np.array(cv_exists), np.array(jd_exists))

    # Inference qua BERT
    cv_batch = [cv_texts[i] for i in range(10) for j in range(5)]
    jd_batch = [jd_texts[j] for i in range(10) for j in range(5)]
    tok = tokenizer(cv_batch, jd_batch, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
    
    model.eval()
    with torch.no_grad():
        cls_emb = encoder(**tok).last_hidden_state[:, 0, :]
        # Gọi model (Unpack đúng 3 giá trị: score, masked_aij, raw_aij)
        final_score, m_aij_tensor, r_aij_tensor = model(cls_emb.unsqueeze(0), 
                                                        torch.tensor(m_ij.flatten(), device=device).unsqueeze(0))
        
        a_raw = r_aij_tensor.squeeze(0).cpu().numpy().reshape(10, 5)
        a_masked = m_aij_tensor.squeeze(0).cpu().numpy().reshape(10, 5)
        score = final_score.item()

    return cv_texts, jd_texts, a_raw, a_masked, score

# ═══════════════════════════════════════════════════════════════════════════
# 3. VẼ BIỂU ĐỒ
# ═══════════════════════════════════════════════════════════════════════════
for row in sample_to_vis:
    cv_t, jd_t, a_raw, a_masked, score = compute_heatmap_data(row)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))
    yticks = [get_clean_label(v, t, is_cv=True)  for v, t in zip(CV_VIEWS, cv_t)]
    xticks = [get_clean_label(v, t, is_cv=False) for v, t in zip(JD_VIEWS, jd_t)]

    # Heatmap 1: Before Masking (Màu xanh - Kết quả BERT học được)
    sns.heatmap(a_raw, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_xticklabels(xticks, rotation=45, ha="right")
    ax1.set_yticklabels(yticks, rotation=0)
    ax1.set_title(f"1. Before Masking (BERT Alignment)\nFile: {row['cv']}", fontsize=14, fontweight='bold')

    # Heatmap 2: After Masking (Màu đỏ - Kết quả đã lọc nhiễu)
    sns.heatmap(a_masked, annot=True, fmt=".2f", cmap="OrRd", ax=ax2, cbar_kws={'shrink': 0.8})
    ax2.set_xticklabels(xticks, rotation=45, ha="right")
    ax2.set_yticklabels(yticks, rotation=0)
    ax2.set_title(f"2. After Masking (Final Logic)\nJD: {row['jd']}", fontsize=14, fontweight='bold')

    plt.suptitle(f"SAMPLE ID: {row['id']} | Ground Truth: {row['score']:.4f} | AI Predicted: {score*100:.2f}%", 
                 fontsize=18, y=1.02, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print("═"*110)

In [ ]:
import os
import torch
import zipfile
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Folder & basename
# =========================
BASE_DIR = "weighted-sum/weighted-sum-keepitreal-vietnamese-sbert-penalty-scoring"
BASENAME = "weighted-sum-keepitreal-vietnamese-sbert-penalty-scoring"

os.makedirs(BASE_DIR, exist_ok=True)

# =========================
# 1. SAVE PLOT (CÓ FINAL LOSS)
# =========================
epochs = len(train_losses)
x = np.arange(1, epochs + 1)

final_train = train_losses[-1]
final_val = val_losses[-1]

plt.figure(figsize=(8, 5))
plt.plot(
    x,
    train_losses,
    label=f"Train Loss (final = {final_train:.4f})"
)
plt.plot(
    x,
    val_losses,
    label=f"Validation Loss (final = {final_val:.4f})"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

plot_path = os.path.join(BASE_DIR, f"{BASENAME}.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"🖼️ Đã lưu plot tại: {plot_path}")

# =========================
# 2. SAVE MODEL (.pth) — FULL WEIGHTS
# =========================
model_path = os.path.join(BASE_DIR, f"{BASENAME}.pth")

torch.save(
    {
        "scorer": model.scorer.state_dict(),
        "fusion": model.fusion.state_dict(),
        "encoder_name": ENCODER_NAME,
        "fusion_name": FUSION_NAME,
    },
    model_path
)

print(f"💾 Đã lưu model tại: {model_path}")

# =========================
# 3. ZIP CẢ FOLDER
# =========================
zip_path = f"{BASE_DIR}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(BASE_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, BASE_DIR)
            z.write(full_path, arcname)

print(f"📦 Đã zip toàn bộ folder tại: {zip_path}")

# MLP

In [ ]:
FUSION_NAME = "baseline_mlp"

class ViewPairScorer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(hidden_dim, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 1))
        nn.init.constant_(self.fc[3].bias, -2.0) 
    def forward(self, cls): return torch.sigmoid(self.fc(cls)).squeeze(-1)

class AdvancedFusionMLP(nn.Module):
    def __init__(self, input_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, a_ij, mask):
        x = torch.cat([a_ij, mask], dim=-1) # 100 features
        return self.net(x)

class FusionWithScorer(nn.Module):
    def __init__(self, hidden_dim, fusion_module):
        super().__init__()
        self.scorer = ViewPairScorer(hidden_dim)
        self.fusion = fusion_module
    def forward(self, cls, mask):
        raw_a_ij = self.scorer(cls) 
        masked_a_ij = raw_a_ij * mask 
        final_score = self.fusion(masked_a_ij, mask)               
        return final_score, masked_a_ij, raw_a_ij                                 

# Loaders & Init
train_loader = DataLoader(CrossEncDataset(X_train, M_train, y_train, b_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(CrossEncDataset(X_val,   M_val,   y_val,   b_val),   batch_size=32)
test_loader  = DataLoader(CrossEncDataset(X_test,  M_test,  y_test,  b_test),  batch_size=32)

model = FusionWithScorer(HIDDEN_DIM, AdvancedFusionMLP(100)).to(device)
criterion_reg, criterion_bin = nn.MSELoss(), nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

print(f"✅ SIMPLE MLP SYNCED READY | Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# =========================
# Training config
# =========================
epochs = 150
best_val_loss = float('inf')

safe_slug = ENCODER_SLUG.replace('/', '_')
CKPT_PATH = f"{CKPT_DIR}/{FUSION_NAME}_{safe_slug}.pth"
os.makedirs(CKPT_DIR, exist_ok=True)

# Hệ số phạt đồng bộ
LAMBDA_PENALTY = 2.0 

# History
train_losses = []
val_losses = []
lrs = []

print(f"🔥 Bắt đầu Training {FUSION_NAME} | Sync Mode (BCE weight: 0.5) | encoder={ENCODER_NAME}")

for epoch in range(epochs):
    # =========================
    # Train
    # =========================
    model.train()
    total_train_loss = 0.0
    
    for cls, mask, by_reg, by_bin in train_loader:
        cls, mask, by_reg, by_bin = cls.to(device), mask.to(device), by_reg.to(device), by_bin.to(device)
        optimizer.zero_grad()
        
        pred, masked_a_ij, raw_a_ij = model(cls, mask)
        
        task_loss = criterion_reg(pred, by_reg) + 0.5 * criterion_bin(pred, by_bin)
        
        empty_penalty = torch.mean(((1 - mask) * raw_a_ij) ** 2)
        
        activity_loss = torch.mean(mask * (0.1 - masked_a_ij).clamp(min=0)) 
        
        loss = task_loss + (LAMBDA_PENALTY * empty_penalty) + (0.5 * activity_loss)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()

    avg_train = total_train_loss / len(train_loader)

    # =========================
    # Validation
    # =========================
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for cls, mask, by_reg, by_bin in val_loader:
            cls, mask, by_reg, by_bin = cls.to(device), mask.to(device), by_reg.to(device), by_bin.to(device)
            out, m_aij, r_aij = model(cls, mask)
            
            # ✅ ĐỒNG BỘ: Tính Val Task Loss với hệ số 0.5
            v_task = criterion_reg(out, by_reg) + 0.5 * criterion_bin(out, by_bin)
            
            # ✅ ĐỒNG BỘ: Tính đầy đủ Penalty và Activity cho Val
            v_pen = torch.mean(((1 - mask) * r_aij) ** 2)
            v_act = torch.mean(mask * (0.1 - m_aij).clamp(min=0))
            
            # Tổng hợp Val Loss (Sử dụng đúng công thức như lúc Train)
            total_val_loss += (v_task + (LAMBDA_PENALTY * v_pen) + (0.5 * v_act)).item()

    avg_val = total_val_loss / len(val_loader)
    scheduler.step(avg_val)

    train_losses.append(avg_train)
    val_losses.append(avg_val)
    lrs.append(optimizer.param_groups[0]['lr'])

    if (epoch + 1) % 10 == 0:
        lr_cur = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:3d} | Train: {avg_train:.6f} | Val: {avg_val:.6f} | LR: {lr_cur:.6e}")

    # Checkpoint (Lưu model tốt nhất dựa trên Val Loss tổng hợp)
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
        torch.save({
            'scorer': model.scorer.state_dict(),
            'fusion': model.fusion.state_dict(),
            'hidden_dim': HIDDEN_DIM,
            'encoder': ENCODER_NAME,
            'epoch': epoch + 1,
            'val_loss': avg_val
        }, CKPT_PATH)

print(f"✅ Hoàn tất Training! Best Val Loss (inc. Penalty): {best_val_loss:.6f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Safety check
# =========================
assert 'train_losses' in globals(), "❌ Chưa có train_losses"
assert 'val_losses' in globals(), "❌ Chưa có val_losses"

# =========================
# Data
# =========================
epochs = len(train_losses)
x = np.arange(1, epochs + 1)

final_train = train_losses[-1]
final_val = val_losses[-1]

# =========================
# Plot
# =========================
plt.figure(figsize=(8, 5))
plt.plot(
    x,
    train_losses,
    label=f"Train Loss (final = {final_train:.4f})"
)
plt.plot(
    x,
    val_losses,
    label=f"Validation Loss (final = {final_val:.4f})"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()      # ✅ chỉ show
plt.close()

In [ ]:
import torch
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from tqdm import tqdm

# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD BEST CHECKPOINT
# ═══════════════════════════════════════════════════════════════════════════
print(f"📂 Loading best model from: {CKPT_PATH}")
checkpoint = torch.load(CKPT_PATH, map_location=device)

model.scorer.load_state_dict(checkpoint['scorer'])
model.fusion.load_state_dict(checkpoint['fusion'])
model.eval()

all_preds = []
all_true_reg = []
all_true_bin = []

# ═══════════════════════════════════════════════════════════════════════════
# 2. TESTING
# ═══════════════════════════════════════════════════════════════════════════
print("🚀 Đang thực hiện dự đoán trên tập Test (Vietnamese MTL)...")
with torch.no_grad():
    for cls, mask, by_reg, by_bin in tqdm(test_loader, desc="Testing"):
        cls, mask = cls.to(device), mask.to(device)

        outputs, _, _ = model(cls, mask)

        all_preds.extend(outputs.cpu().numpy().flatten())
        all_true_reg.extend(by_reg.numpy().flatten())
        all_true_bin.extend(by_bin.numpy().flatten())

all_preds = np.array(all_preds)
all_true_reg = np.array(all_true_reg)
all_true_bin = np.array(all_true_bin)

# Sanity check
print(
    f"🔎 Sanity | preds: mean={all_preds.mean():.4f} "
    f"std={all_preds.std():.4f} "
    f"min={all_preds.min():.4f} "
    f"max={all_preds.max():.4f}"
)

# Binary threshold
threshold = 0.5
all_preds_bin = (all_preds >= threshold).astype(int)

# ═══════════════════════════════════════════════════════════════════════════
# 3. METRICS
# ═══════════════════════════════════════════════════════════════════════════
mae = mean_absolute_error(all_true_reg, all_preds)
mse = mean_squared_error(all_true_reg, all_preds)

acc = accuracy_score(all_true_bin, all_preds_bin)
pre = precision_score(all_true_bin, all_preds_bin, zero_division=0)
rec = recall_score(all_true_bin, all_preds_bin, zero_division=0)
f1  = f1_score(all_true_bin, all_preds_bin, zero_division=0)

print("\n" + "="*60)
print("📊 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (VIETNAMESE)")
print("-" * 60)
print(f"Encoder: {ENCODER_NAME}")
print(f"Fusion : {FUSION_NAME}")
print("-" * 60)
print(f"{'MAE':<25} | {mae:.4f}")
print(f"{'MSE':<25} | {mse:.4f}")
print(f"{'Accuracy':<25} | {acc*100:.2f}%")
print(f"{'Precision':<25} | {pre:.4f}")
print(f"{'Recall':<25} | {rec:.4f}")
print(f"{'F1-Score':<25} | {f1:.4f}")
print("="*60)

# ═══════════════════════════════════════════════════════════════════════════
# 4. HIỂN THỊ ĐỐI CHIẾU 10 MẪU ĐẦU TIÊN
# ═══════════════════════════════════════════════════════════════════════════
print("\n🔍 Đối chiếu mẫu thực tế:")
print(f"{'ID':<4} | {'Thực tế':<10} | {'Điểm Gốc':<10} | {'AI Dự đoán':<10} | {'Kết luận'}")
print("-" * 65)
for i in range(min(10, len(all_preds))):
    label_true = "Match" if all_true_bin[i] == 1 else "Mismatch"
    label_pred = "Match" if all_preds_bin[i] == 1 else "Mismatch"
    status = "✅" if all_true_bin[i] == all_preds_bin[i] else "❌"
    print(f"{i:<4} | {label_true:<10} | {all_true_reg[i]:<10.4f} | {all_preds[i]:<10.4f} | {status} {label_pred}")
    

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import json
import os
import numpy as np
import torch

# ═══════════════════════════════════════════════════════════════════════════
# 1. CHUẨN BỊ DỮ LIỆU (Sử dụng 3 mẫu từ tập Test Match)
# ═══════════════════════════════════════════════════════════════════════════
test_match_df = pd.read_csv(os.path.join(SPLIT_ROOT, "test/match/test_match.csv"))
# Lấy 3 mẫu ngẫu nhiên để vẽ
sample_to_vis = test_match_df.sample(3, random_state=42).to_dict('records')

def get_clean_label(view_name, text, is_cv=True):
    if not text or text == "[EMPTY_SECTION]":
        return f"[{view_name[:4].upper()}]\n(Trống)"
    limit = 35
    short = (text[:limit] + '..') if len(text) > limit else text
    return f"[{view_name[:4].upper()}] {short}" if is_cv else f"[{view_name[:4].upper()}]\n{short}"

# ═══════════════════════════════════════════════════════════════════════════
# 2. HÀM TÍNH TOÁN ALIGNMENT CHI TIẾT
# ═══════════════════════════════════════════════════════════════════════════
def compute_heatmap_data(row):
    # Nạp JSON (Sử dụng đúng logic fallback của bạn)
    with open(os.path.join(CV_DIR, row['cv']), 'r', encoding='utf-8-sig') as f:
        cv_obj = json.load(f)
        cv_j = cv_obj.get('resume_structured', cv_obj)
    with open(os.path.join(JD_DIR, row['jd']), 'r', encoding='utf-8-sig') as f:
        jd_obj = json.load(f)
        jd_j = jd_obj.get('job_description_structured', jd_obj)

    # Trích xuất views
    cv_texts, cv_exists = zip(*[process_view(cv_j.get(v)) for v in CV_VIEWS])
    jd_texts, jd_exists = zip(*[process_view(jd_j.get(v)) for v in JD_VIEWS])
    m_ij = np.outer(np.array(cv_exists), np.array(jd_exists))

    # Inference qua BERT
    cv_batch = [cv_texts[i] for i in range(10) for j in range(5)]
    jd_batch = [jd_texts[j] for i in range(10) for j in range(5)]
    tok = tokenizer(cv_batch, jd_batch, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
    
    model.eval()
    with torch.no_grad():
        cls_emb = encoder(**tok).last_hidden_state[:, 0, :]
        # Gọi model (Unpack đúng 3 giá trị: score, masked_aij, raw_aij)
        final_score, m_aij_tensor, r_aij_tensor = model(cls_emb.unsqueeze(0), 
                                                        torch.tensor(m_ij.flatten(), device=device).unsqueeze(0))
        
        a_raw = r_aij_tensor.squeeze(0).cpu().numpy().reshape(10, 5)
        a_masked = m_aij_tensor.squeeze(0).cpu().numpy().reshape(10, 5)
        score = final_score.item()

    return cv_texts, jd_texts, a_raw, a_masked, score

# ═══════════════════════════════════════════════════════════════════════════
# 3. VẼ BIỂU ĐỒ
# ═══════════════════════════════════════════════════════════════════════════
for row in sample_to_vis:
    cv_t, jd_t, a_raw, a_masked, score = compute_heatmap_data(row)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))
    yticks = [get_clean_label(v, t, is_cv=True)  for v, t in zip(CV_VIEWS, cv_t)]
    xticks = [get_clean_label(v, t, is_cv=False) for v, t in zip(JD_VIEWS, jd_t)]

    # Heatmap 1: Before Masking (Màu xanh - Kết quả BERT học được)
    sns.heatmap(a_raw, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_xticklabels(xticks, rotation=45, ha="right")
    ax1.set_yticklabels(yticks, rotation=0)
    ax1.set_title(f"1. Before Masking (BERT Alignment)\nFile: {row['cv']}", fontsize=14, fontweight='bold')

    # Heatmap 2: After Masking (Màu đỏ - Kết quả đã lọc nhiễu)
    sns.heatmap(a_masked, annot=True, fmt=".2f", cmap="OrRd", ax=ax2, cbar_kws={'shrink': 0.8})
    ax2.set_xticklabels(xticks, rotation=45, ha="right")
    ax2.set_yticklabels(yticks, rotation=0)
    ax2.set_title(f"2. After Masking (Final Logic)\nJD: {row['jd']}", fontsize=14, fontweight='bold')

    plt.suptitle(f"SAMPLE ID: {row['id']} | Ground Truth: {row['score']:.4f} | AI Predicted: {score*100:.2f}%", 
                 fontsize=18, y=1.02, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print("═"*110)

In [ ]:
import os
import torch
import zipfile
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Folder & basename
# =========================
BASE_DIR = "mlp/mlp-keepitreal-vietnamese-sbert-penalty-scoring"
BASENAME = "mlp-keepitreal-vietnamese-sbert-penalty-scoring"

os.makedirs(BASE_DIR, exist_ok=True)

# =========================
# 1. SAVE PLOT (CÓ FINAL LOSS)
# =========================
epochs = len(train_losses)
x = np.arange(1, epochs + 1)

final_train = train_losses[-1]
final_val = val_losses[-1]

plt.figure(figsize=(8, 5))
plt.plot(
    x,
    train_losses,
    label=f"Train Loss (final = {final_train:.4f})"
)
plt.plot(
    x,
    val_losses,
    label=f"Validation Loss (final = {final_val:.4f})"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

plot_path = os.path.join(BASE_DIR, f"{BASENAME}.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"🖼️ Đã lưu plot tại: {plot_path}")

# =========================
# 2. SAVE MODEL (.pth) — FULL WEIGHTS
# =========================
model_path = os.path.join(BASE_DIR, f"{BASENAME}.pth")

torch.save(
    {
        "scorer": model.scorer.state_dict(),
        "fusion": model.fusion.state_dict(),
        "encoder_name": ENCODER_NAME,
        "fusion_name": FUSION_NAME,
    },
    model_path
)

print(f"💾 Đã lưu model tại: {model_path}")

# =========================
# 3. ZIP CẢ FOLDER
# =========================
zip_path = f"{BASE_DIR}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(BASE_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, BASE_DIR)
            z.write(full_path, arcname)

print(f"📦 Đã zip toàn bộ folder tại: {zip_path}")

# MLP Residual

In [ ]:
FUSION_NAME = "residual_mlp"

class ViewPairScorer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(hidden_dim, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 1))
        nn.init.constant_(self.fc[3].bias, -2.0) 
    def forward(self, cls): return torch.sigmoid(self.fc(cls)).squeeze(-1)

class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.ln = nn.LayerNorm(dim); self.fc = nn.Linear(dim, dim)
    def forward(self, x): return x + F.relu(self.fc(self.ln(x)))

class AdvancedFusionResidualMLP(nn.Module):
    def __init__(self, input_dim=100):
        super().__init__()
        self.input_layer = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU())
        self.res1, self.res2 = ResidualBlock(256), ResidualBlock(256)
        self.fc_hidden = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 64), nn.ReLU())
        self.output_layer = nn.Sequential(nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, a_ij, mask):
        x = torch.cat([a_ij, mask], dim=-1)
        x = self.input_layer(x); x = self.res1(x); x = self.res2(x); x = self.fc_hidden(x)
        return self.output_layer(x)

class FusionWithScorer(nn.Module):
    def __init__(self, hidden_dim, fusion_module):
        super().__init__()
        self.scorer = ViewPairScorer(hidden_dim)
        self.fusion = fusion_module
    def forward(self, cls, mask):
        raw_a_ij = self.scorer(cls) 
        masked_a_ij = raw_a_ij * mask 
        final_score = self.fusion(masked_a_ij, mask)               
        return final_score, masked_a_ij, raw_a_ij

# Loaders & Init
train_loader = DataLoader(CrossEncDataset(X_train, M_train, y_train, b_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(CrossEncDataset(X_val,   M_val,   y_val,   b_val),   batch_size=32)
test_loader  = DataLoader(CrossEncDataset(X_test,  M_test,  y_test,  b_test),  batch_size=32)

model = FusionWithScorer(HIDDEN_DIM, AdvancedFusionResidualMLP(100)).to(device)
criterion_reg, criterion_bin = nn.MSELoss(), nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

print(f"✅ RESIDUAL MLP SYNCED READY | Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# =========================
# Training config
# =========================
epochs = 150
best_val_loss = float('inf')

safe_slug = ENCODER_SLUG.replace('/', '_')
CKPT_PATH = f"{CKPT_DIR}/{FUSION_NAME}_{safe_slug}.pth"
os.makedirs(CKPT_DIR, exist_ok=True)

# Hệ số phạt đồng bộ
LAMBDA_PENALTY = 2.0 

# History
train_losses = []
val_losses = []
lrs = []

print(f"🔥 Bắt đầu Training {FUSION_NAME} | Sync Mode (BCE weight: 0.5) | encoder={ENCODER_NAME}")

for epoch in range(epochs):
    # =========================
    # Train
    # =========================
    model.train()
    total_train_loss = 0.0
    
    for cls, mask, by_reg, by_bin in train_loader:
        cls, mask, by_reg, by_bin = cls.to(device), mask.to(device), by_reg.to(device), by_bin.to(device)
        optimizer.zero_grad()
        
        pred, masked_a_ij, raw_a_ij = model(cls, mask)
        
        task_loss = criterion_reg(pred, by_reg) + 0.5 * criterion_bin(pred, by_bin)
        
        empty_penalty = torch.mean(((1 - mask) * raw_a_ij) ** 2)
        
        activity_loss = torch.mean(mask * (0.1 - masked_a_ij).clamp(min=0)) 
        
        loss = task_loss + (LAMBDA_PENALTY * empty_penalty) + (0.5 * activity_loss)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()

    avg_train = total_train_loss / len(train_loader)

    # =========================
    # Validation
    # =========================
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for cls, mask, by_reg, by_bin in val_loader:
            cls, mask, by_reg, by_bin = cls.to(device), mask.to(device), by_reg.to(device), by_bin.to(device)
            out, m_aij, r_aij = model(cls, mask)
            
            # ✅ ĐỒNG BỘ: Tính Val Task Loss với hệ số 0.5
            v_task = criterion_reg(out, by_reg) + 0.5 * criterion_bin(out, by_bin)
            
            # ✅ ĐỒNG BỘ: Tính đầy đủ Penalty và Activity cho Val
            v_pen = torch.mean(((1 - mask) * r_aij) ** 2)
            v_act = torch.mean(mask * (0.1 - m_aij).clamp(min=0))
            
            # Tổng hợp Val Loss (Sử dụng đúng công thức như lúc Train)
            total_val_loss += (v_task + (LAMBDA_PENALTY * v_pen) + (0.5 * v_act)).item()

    avg_val = total_val_loss / len(val_loader)
    scheduler.step(avg_val)

    train_losses.append(avg_train)
    val_losses.append(avg_val)
    lrs.append(optimizer.param_groups[0]['lr'])

    if (epoch + 1) % 10 == 0:
        lr_cur = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:3d} | Train: {avg_train:.6f} | Val: {avg_val:.6f} | LR: {lr_cur:.6e}")

    # Checkpoint (Lưu model tốt nhất dựa trên Val Loss tổng hợp)
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
        torch.save({
            'scorer': model.scorer.state_dict(),
            'fusion': model.fusion.state_dict(),
            'hidden_dim': HIDDEN_DIM,
            'encoder': ENCODER_NAME,
            'epoch': epoch + 1,
            'val_loss': avg_val
        }, CKPT_PATH)

print(f"✅ Hoàn tất Training! Best Val Loss (inc. Penalty): {best_val_loss:.6f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Safety check
# =========================
assert 'train_losses' in globals(), "❌ Chưa có train_losses"
assert 'val_losses' in globals(), "❌ Chưa có val_losses"

# =========================
# Data
# =========================
epochs = len(train_losses)
x = np.arange(1, epochs + 1)

final_train = train_losses[-1]
final_val = val_losses[-1]

# =========================
# Plot
# =========================
plt.figure(figsize=(8, 5))
plt.plot(
    x,
    train_losses,
    label=f"Train Loss (final = {final_train:.4f})"
)
plt.plot(
    x,
    val_losses,
    label=f"Validation Loss (final = {final_val:.4f})"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()      # ✅ chỉ show
plt.close()

In [ ]:
import torch
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from tqdm import tqdm

# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD BEST CHECKPOINT
# ═══════════════════════════════════════════════════════════════════════════
print(f"📂 Loading best model from: {CKPT_PATH}")
checkpoint = torch.load(CKPT_PATH, map_location=device)

model.scorer.load_state_dict(checkpoint['scorer'])
model.fusion.load_state_dict(checkpoint['fusion'])
model.eval()

all_preds = []
all_true_reg = []
all_true_bin = []

# ═══════════════════════════════════════════════════════════════════════════
# 2. TESTING
# ═══════════════════════════════════════════════════════════════════════════
print("🚀 Đang thực hiện dự đoán trên tập Test (Vietnamese MTL)...")
with torch.no_grad():
    for cls, mask, by_reg, by_bin in tqdm(test_loader, desc="Testing"):
        cls, mask = cls.to(device), mask.to(device)

        outputs, _, _ = model(cls, mask)

        all_preds.extend(outputs.cpu().numpy().flatten())
        all_true_reg.extend(by_reg.numpy().flatten())
        all_true_bin.extend(by_bin.numpy().flatten())

all_preds = np.array(all_preds)
all_true_reg = np.array(all_true_reg)
all_true_bin = np.array(all_true_bin)

# Sanity check
print(
    f"🔎 Sanity | preds: mean={all_preds.mean():.4f} "
    f"std={all_preds.std():.4f} "
    f"min={all_preds.min():.4f} "
    f"max={all_preds.max():.4f}"
)

# Binary threshold
threshold = 0.5
all_preds_bin = (all_preds >= threshold).astype(int)

# ═══════════════════════════════════════════════════════════════════════════
# 3. METRICS
# ═══════════════════════════════════════════════════════════════════════════
mae = mean_absolute_error(all_true_reg, all_preds)
mse = mean_squared_error(all_true_reg, all_preds)

acc = accuracy_score(all_true_bin, all_preds_bin)
pre = precision_score(all_true_bin, all_preds_bin, zero_division=0)
rec = recall_score(all_true_bin, all_preds_bin, zero_division=0)
f1  = f1_score(all_true_bin, all_preds_bin, zero_division=0)

print("\n" + "="*60)
print("📊 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (VIETNAMESE)")
print("-" * 60)
print(f"Encoder: {ENCODER_NAME}")
print(f"Fusion : {FUSION_NAME}")
print("-" * 60)
print(f"{'MAE':<25} | {mae:.4f}")
print(f"{'MSE':<25} | {mse:.4f}")
print(f"{'Accuracy':<25} | {acc*100:.2f}%")
print(f"{'Precision':<25} | {pre:.4f}")
print(f"{'Recall':<25} | {rec:.4f}")
print(f"{'F1-Score':<25} | {f1:.4f}")
print("="*60)

# ═══════════════════════════════════════════════════════════════════════════
# 4. HIỂN THỊ ĐỐI CHIẾU 10 MẪU ĐẦU TIÊN
# ═══════════════════════════════════════════════════════════════════════════
print("\n🔍 Đối chiếu mẫu thực tế:")
print(f"{'ID':<4} | {'Thực tế':<10} | {'Điểm Gốc':<10} | {'AI Dự đoán':<10} | {'Kết luận'}")
print("-" * 65)
for i in range(min(10, len(all_preds))):
    label_true = "Match" if all_true_bin[i] == 1 else "Mismatch"
    label_pred = "Match" if all_preds_bin[i] == 1 else "Mismatch"
    status = "✅" if all_true_bin[i] == all_preds_bin[i] else "❌"
    print(f"{i:<4} | {label_true:<10} | {all_true_reg[i]:<10.4f} | {all_preds[i]:<10.4f} | {status} {label_pred}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import json
import os
import numpy as np
import torch

# ═══════════════════════════════════════════════════════════════════════════
# 1. CHUẨN BỊ DỮ LIỆU (Sử dụng 3 mẫu từ tập Test Match)
# ═══════════════════════════════════════════════════════════════════════════
test_match_df = pd.read_csv(os.path.join(SPLIT_ROOT, "test/match/test_match.csv"))
# Lấy 3 mẫu ngẫu nhiên để vẽ
sample_to_vis = test_match_df.sample(3, random_state=42).to_dict('records')

def get_clean_label(view_name, text, is_cv=True):
    if not text or text == "[EMPTY_SECTION]":
        return f"[{view_name[:4].upper()}]\n(Trống)"
    limit = 35
    short = (text[:limit] + '..') if len(text) > limit else text
    return f"[{view_name[:4].upper()}] {short}" if is_cv else f"[{view_name[:4].upper()}]\n{short}"

# ═══════════════════════════════════════════════════════════════════════════
# 2. HÀM TÍNH TOÁN ALIGNMENT CHI TIẾT
# ═══════════════════════════════════════════════════════════════════════════
def compute_heatmap_data(row):
    # Nạp JSON (Sử dụng đúng logic fallback của bạn)
    with open(os.path.join(CV_DIR, row['cv']), 'r', encoding='utf-8-sig') as f:
        cv_obj = json.load(f)
        cv_j = cv_obj.get('resume_structured', cv_obj)
    with open(os.path.join(JD_DIR, row['jd']), 'r', encoding='utf-8-sig') as f:
        jd_obj = json.load(f)
        jd_j = jd_obj.get('job_description_structured', jd_obj)

    # Trích xuất views
    cv_texts, cv_exists = zip(*[process_view(cv_j.get(v)) for v in CV_VIEWS])
    jd_texts, jd_exists = zip(*[process_view(jd_j.get(v)) for v in JD_VIEWS])
    m_ij = np.outer(np.array(cv_exists), np.array(jd_exists))

    # Inference qua BERT
    cv_batch = [cv_texts[i] for i in range(10) for j in range(5)]
    jd_batch = [jd_texts[j] for i in range(10) for j in range(5)]
    tok = tokenizer(cv_batch, jd_batch, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
    
    model.eval()
    with torch.no_grad():
        cls_emb = encoder(**tok).last_hidden_state[:, 0, :]
        # Gọi model (Unpack đúng 3 giá trị: score, masked_aij, raw_aij)
        final_score, m_aij_tensor, r_aij_tensor = model(cls_emb.unsqueeze(0), 
                                                        torch.tensor(m_ij.flatten(), device=device).unsqueeze(0))
        
        a_raw = r_aij_tensor.squeeze(0).cpu().numpy().reshape(10, 5)
        a_masked = m_aij_tensor.squeeze(0).cpu().numpy().reshape(10, 5)
        score = final_score.item()

    return cv_texts, jd_texts, a_raw, a_masked, score

# ═══════════════════════════════════════════════════════════════════════════
# 3. VẼ BIỂU ĐỒ
# ═══════════════════════════════════════════════════════════════════════════
for row in sample_to_vis:
    cv_t, jd_t, a_raw, a_masked, score = compute_heatmap_data(row)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))
    yticks = [get_clean_label(v, t, is_cv=True)  for v, t in zip(CV_VIEWS, cv_t)]
    xticks = [get_clean_label(v, t, is_cv=False) for v, t in zip(JD_VIEWS, jd_t)]

    # Heatmap 1: Before Masking (Màu xanh - Kết quả BERT học được)
    sns.heatmap(a_raw, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_xticklabels(xticks, rotation=45, ha="right")
    ax1.set_yticklabels(yticks, rotation=0)
    ax1.set_title(f"1. Before Masking (BERT Alignment)\nFile: {row['cv']}", fontsize=14, fontweight='bold')

    # Heatmap 2: After Masking (Màu đỏ - Kết quả đã lọc nhiễu)
    sns.heatmap(a_masked, annot=True, fmt=".2f", cmap="OrRd", ax=ax2, cbar_kws={'shrink': 0.8})
    ax2.set_xticklabels(xticks, rotation=45, ha="right")
    ax2.set_yticklabels(yticks, rotation=0)
    ax2.set_title(f"2. After Masking (Final Logic)\nJD: {row['jd']}", fontsize=14, fontweight='bold')

    plt.suptitle(f"SAMPLE ID: {row['id']} | Ground Truth: {row['score']:.4f} | AI Predicted: {score*100:.2f}%", 
                 fontsize=18, y=1.02, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print("═"*110)

In [ ]:
import os
import torch
import zipfile
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Folder & basename
# =========================
BASE_DIR = "mlp-residual/mlp-residual-keepitreal-vietnamese-sbert-penalty-scoring"
BASENAME = "mlp-residual-keepitreal-vietnamese-sbert-penalty-scoring"

os.makedirs(BASE_DIR, exist_ok=True)

# =========================
# 1. SAVE PLOT (CÓ FINAL LOSS)
# =========================
epochs = len(train_losses)
x = np.arange(1, epochs + 1)

final_train = train_losses[-1]
final_val = val_losses[-1]

plt.figure(figsize=(8, 5))
plt.plot(
    x,
    train_losses,
    label=f"Train Loss (final = {final_train:.4f})"
)
plt.plot(
    x,
    val_losses,
    label=f"Validation Loss (final = {final_val:.4f})"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

plot_path = os.path.join(BASE_DIR, f"{BASENAME}.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"🖼️ Đã lưu plot tại: {plot_path}")

# =========================
# 2. SAVE MODEL (.pth) — FULL WEIGHTS
# =========================
model_path = os.path.join(BASE_DIR, f"{BASENAME}.pth")

torch.save(
    {
        "scorer": model.scorer.state_dict(),
        "fusion": model.fusion.state_dict(),
        "encoder_name": ENCODER_NAME,
        "fusion_name": FUSION_NAME,
    },
    model_path
)

print(f"💾 Đã lưu model tại: {model_path}")

# =========================
# 3. ZIP CẢ FOLDER
# =========================
zip_path = f"{BASE_DIR}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(BASE_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, BASE_DIR)
            z.write(full_path, arcname)

print(f"📦 Đã zip toàn bộ folder tại: {zip_path}")

# Attention

In [ ]:
FUSION_NAME = "attention"

class ViewPairScorer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(hidden_dim, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 1))
        nn.init.constant_(self.fc[3].bias, -2.0) 
    def forward(self, cls): return torch.sigmoid(self.fc(cls)).squeeze(-1)

class AttentionFusionModel(nn.Module):
    def __init__(self, num_views=50, d_model=64, nhead=8):
        super().__init__()
        self.input_projection = nn.Linear(2, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.2, batch_first=True)
        self.attention_block = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.fc_fusion = nn.Sequential(nn.Linear(d_model * num_views, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1), nn.Sigmoid())
    def forward(self, a_ij, mask):
        batch_size = a_ij.shape[0]
        x = torch.stack([a_ij, mask], dim=-1)
        h = self.input_projection(x); attn_out = self.attention_block(h)
        return self.fc_fusion(attn_out.reshape(batch_size, -1))

class FusionWithScorer(nn.Module):
    def __init__(self, hidden_dim, fusion_module):
        super().__init__()
        self.scorer = ViewPairScorer(hidden_dim)
        self.fusion = fusion_module
    def forward(self, cls, mask):
        raw_a_ij = self.scorer(cls) 
        masked_a_ij = raw_a_ij * mask 
        final_score = self.fusion(masked_a_ij, mask)               
        return final_score, masked_a_ij, raw_a_ij

# Loaders & Init
train_loader = DataLoader(CrossEncDataset(X_train, M_train, y_train, b_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(CrossEncDataset(X_val,   M_val,   y_val,   b_val),   batch_size=32)
test_loader  = DataLoader(CrossEncDataset(X_test,  M_test,  y_test,  b_test),  batch_size=32)

model = FusionWithScorer(HIDDEN_DIM, AttentionFusionModel(50)).to(device)
criterion_reg, criterion_bin = nn.MSELoss(), nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

print(f"✅ ATTENTION SYNCED READY | Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# =========================
# Training config
# =========================
epochs = 150
best_val_loss = float('inf')

safe_slug = ENCODER_SLUG.replace('/', '_')
CKPT_PATH = f"{CKPT_DIR}/{FUSION_NAME}_{safe_slug}.pth"
os.makedirs(CKPT_DIR, exist_ok=True)

# Hệ số phạt đồng bộ
LAMBDA_PENALTY = 2.0 

# History
train_losses = []
val_losses = []
lrs = []

print(f"🔥 Bắt đầu Training {FUSION_NAME} | Sync Mode (BCE weight: 0.5) | encoder={ENCODER_NAME}")

for epoch in range(epochs):
    # =========================
    # Train
    # =========================
    model.train()
    total_train_loss = 0.0
    
    for cls, mask, by_reg, by_bin in train_loader:
        cls, mask, by_reg, by_bin = cls.to(device), mask.to(device), by_reg.to(device), by_bin.to(device)
        optimizer.zero_grad()
        
        pred, masked_a_ij, raw_a_ij = model(cls, mask)
        
        task_loss = criterion_reg(pred, by_reg) + 0.5 * criterion_bin(pred, by_bin)
        
        empty_penalty = torch.mean(((1 - mask) * raw_a_ij) ** 2)
        
        activity_loss = torch.mean(mask * (0.1 - masked_a_ij).clamp(min=0)) 
        
        # Tổng hợp loss huấn luyện
        loss = task_loss + (LAMBDA_PENALTY * empty_penalty) + (0.5 * activity_loss)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()

    avg_train = total_train_loss / len(train_loader)

    # =========================
    # Validation
    # =========================
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for cls, mask, by_reg, by_bin in val_loader:
            cls, mask, by_reg, by_bin = cls.to(device), mask.to(device), by_reg.to(device), by_bin.to(device)
            out, m_aij, r_aij = model(cls, mask)
            
            # ✅ ĐỒNG BỘ: Tính Val Task Loss với hệ số 0.5
            v_task = criterion_reg(out, by_reg) + 0.5 * criterion_bin(out, by_bin)
            
            # ✅ ĐỒNG BỘ: Tính đầy đủ Penalty và Activity cho Val
            v_pen = torch.mean(((1 - mask) * r_aij) ** 2)
            v_act = torch.mean(mask * (0.1 - m_aij).clamp(min=0))
            
            # Tổng hợp Val Loss (Sử dụng đúng công thức như lúc Train)
            total_val_loss += (v_task + (LAMBDA_PENALTY * v_pen) + (0.5 * v_act)).item()

    avg_val = total_val_loss / len(val_loader)
    scheduler.step(avg_val)

    train_losses.append(avg_train)
    val_losses.append(avg_val)
    lrs.append(optimizer.param_groups[0]['lr'])

    if (epoch + 1) % 10 == 0:
        lr_cur = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:3d} | Train: {avg_train:.6f} | Val: {avg_val:.6f} | LR: {lr_cur:.6e}")

    # Checkpoint (Lưu model tốt nhất dựa trên Val Loss tổng hợp)
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
        torch.save({
            'scorer': model.scorer.state_dict(),
            'fusion': model.fusion.state_dict(),
            'hidden_dim': HIDDEN_DIM,
            'encoder': ENCODER_NAME,
            'epoch': epoch + 1,
            'val_loss': avg_val
        }, CKPT_PATH)

print(f"✅ Hoàn tất Training! Best Val Loss (inc. Penalty): {best_val_loss:.6f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Safety check
# =========================
assert 'train_losses' in globals(), "❌ Chưa có train_losses"
assert 'val_losses' in globals(), "❌ Chưa có val_losses"

# =========================
# Data
# =========================
epochs = len(train_losses)
x = np.arange(1, epochs + 1)

final_train = train_losses[-1]
final_val = val_losses[-1]

# =========================
# Plot
# =========================
plt.figure(figsize=(8, 5))
plt.plot(
    x,
    train_losses,
    label=f"Train Loss (final = {final_train:.4f})"
)
plt.plot(
    x,
    val_losses,
    label=f"Validation Loss (final = {final_val:.4f})"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()      # ✅ chỉ show
plt.close()

In [ ]:
import torch
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from tqdm import tqdm

# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD BEST CHECKPOINT
# ═══════════════════════════════════════════════════════════════════════════
print(f"📂 Loading best model from: {CKPT_PATH}")
checkpoint = torch.load(CKPT_PATH, map_location=device)

model.scorer.load_state_dict(checkpoint['scorer'])
model.fusion.load_state_dict(checkpoint['fusion'])
model.eval()

all_preds = []
all_true_reg = []
all_true_bin = []

# ═══════════════════════════════════════════════════════════════════════════
# 2. TESTING
# ═══════════════════════════════════════════════════════════════════════════
print("🚀 Đang thực hiện dự đoán trên tập Test (Vietnamese MTL)...")
with torch.no_grad():
    for cls, mask, by_reg, by_bin in tqdm(test_loader, desc="Testing"):
        cls, mask = cls.to(device), mask.to(device)

        outputs, _, _ = model(cls, mask)

        all_preds.extend(outputs.cpu().numpy().flatten())
        all_true_reg.extend(by_reg.numpy().flatten())
        all_true_bin.extend(by_bin.numpy().flatten())

all_preds = np.array(all_preds)
all_true_reg = np.array(all_true_reg)
all_true_bin = np.array(all_true_bin)

# Sanity check
print(
    f"🔎 Sanity | preds: mean={all_preds.mean():.4f} "
    f"std={all_preds.std():.4f} "
    f"min={all_preds.min():.4f} "
    f"max={all_preds.max():.4f}"
)

# Binary threshold
threshold = 0.5
all_preds_bin = (all_preds >= threshold).astype(int)

# ═══════════════════════════════════════════════════════════════════════════
# 3. METRICS
# ═══════════════════════════════════════════════════════════════════════════
mae = mean_absolute_error(all_true_reg, all_preds)
mse = mean_squared_error(all_true_reg, all_preds)

acc = accuracy_score(all_true_bin, all_preds_bin)
pre = precision_score(all_true_bin, all_preds_bin, zero_division=0)
rec = recall_score(all_true_bin, all_preds_bin, zero_division=0)
f1  = f1_score(all_true_bin, all_preds_bin, zero_division=0)

print("\n" + "="*60)
print("📊 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (VIETNAMESE)")
print("-" * 60)
print(f"Encoder: {ENCODER_NAME}")
print(f"Fusion : {FUSION_NAME}")
print("-" * 60)
print(f"{'MAE':<25} | {mae:.4f}")
print(f"{'MSE':<25} | {mse:.4f}")
print(f"{'Accuracy':<25} | {acc*100:.2f}%")
print(f"{'Precision':<25} | {pre:.4f}")
print(f"{'Recall':<25} | {rec:.4f}")
print(f"{'F1-Score':<25} | {f1:.4f}")
print("="*60)

# ═══════════════════════════════════════════════════════════════════════════
# 4. HIỂN THỊ ĐỐI CHIẾU 10 MẪU ĐẦU TIÊN
# ═══════════════════════════════════════════════════════════════════════════
print("\n🔍 Đối chiếu mẫu thực tế:")
print(f"{'ID':<4} | {'Thực tế':<10} | {'Điểm Gốc':<10} | {'AI Dự đoán':<10} | {'Kết luận'}")
print("-" * 65)
for i in range(min(10, len(all_preds))):
    label_true = "Match" if all_true_bin[i] == 1 else "Mismatch"
    label_pred = "Match" if all_preds_bin[i] == 1 else "Mismatch"
    status = "✅" if all_true_bin[i] == all_preds_bin[i] else "❌"
    print(f"{i:<4} | {label_true:<10} | {all_true_reg[i]:<10.4f} | {all_preds[i]:<10.4f} | {status} {label_pred}")


In [ ]:
import os
import torch
import zipfile
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Folder & basename
# =========================
BASE_DIR = "attention/attention-keepitreal-vietnamese-sbert-penalty-scoring"
BASENAME = "attention-keepitreal-vietnamese-sbert-penalty-scoring"

os.makedirs(BASE_DIR, exist_ok=True)

# =========================
# 1. SAVE PLOT (CÓ FINAL LOSS)
# =========================
epochs = len(train_losses)
x = np.arange(1, epochs + 1)

final_train = train_losses[-1]
final_val = val_losses[-1]

plt.figure(figsize=(8, 5))
plt.plot(
    x,
    train_losses,
    label=f"Train Loss (final = {final_train:.4f})"
)
plt.plot(
    x,
    val_losses,
    label=f"Validation Loss (final = {final_val:.4f})"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

plot_path = os.path.join(BASE_DIR, f"{BASENAME}.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"🖼️ Đã lưu plot tại: {plot_path}")

# =========================
# 2. SAVE MODEL (.pth) — FULL WEIGHTS
# =========================
model_path = os.path.join(BASE_DIR, f"{BASENAME}.pth")

torch.save(
    {
        "scorer": model.scorer.state_dict(),
        "fusion": model.fusion.state_dict(),
        "encoder_name": ENCODER_NAME,
        "fusion_name": FUSION_NAME,
    },
    model_path
)

print(f"💾 Đã lưu model tại: {model_path}")

# =========================
# 3. ZIP CẢ FOLDER
# =========================
zip_path = f"{BASE_DIR}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(BASE_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, BASE_DIR)
            z.write(full_path, arcname)

print(f"📦 Đã zip toàn bộ folder tại: {zip_path}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import json
import os
import numpy as np
import torch

# ═══════════════════════════════════════════════════════════════════════════
# 1. CHUẨN BỊ DỮ LIỆU (Sử dụng 3 mẫu từ tập Test Match)
# ═══════════════════════════════════════════════════════════════════════════
test_match_df = pd.read_csv(os.path.join(SPLIT_ROOT, "test/match/test_match.csv"))
# Lấy 3 mẫu ngẫu nhiên để vẽ
sample_to_vis = test_match_df.sample(3, random_state=42).to_dict('records')

def get_clean_label(view_name, text, is_cv=True):
    if not text or text == "[EMPTY_SECTION]":
        return f"[{view_name[:4].upper()}]\n(Trống)"
    limit = 35
    short = (text[:limit] + '..') if len(text) > limit else text
    return f"[{view_name[:4].upper()}] {short}" if is_cv else f"[{view_name[:4].upper()}]\n{short}"

# ═══════════════════════════════════════════════════════════════════════════
# 2. HÀM TÍNH TOÁN ALIGNMENT CHI TIẾT
# ═══════════════════════════════════════════════════════════════════════════
def compute_heatmap_data(row):
    # Nạp JSON (Sử dụng đúng logic fallback của bạn)
    with open(os.path.join(CV_DIR, row['cv']), 'r', encoding='utf-8-sig') as f:
        cv_obj = json.load(f)
        cv_j = cv_obj.get('resume_structured', cv_obj)
    with open(os.path.join(JD_DIR, row['jd']), 'r', encoding='utf-8-sig') as f:
        jd_obj = json.load(f)
        jd_j = jd_obj.get('job_description_structured', jd_obj)

    # Trích xuất views
    cv_texts, cv_exists = zip(*[process_view(cv_j.get(v)) for v in CV_VIEWS])
    jd_texts, jd_exists = zip(*[process_view(jd_j.get(v)) for v in JD_VIEWS])
    m_ij = np.outer(np.array(cv_exists), np.array(jd_exists))

    # Inference qua BERT
    cv_batch = [cv_texts[i] for i in range(10) for j in range(5)]
    jd_batch = [jd_texts[j] for i in range(10) for j in range(5)]
    tok = tokenizer(cv_batch, jd_batch, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
    
    model.eval()
    with torch.no_grad():
        cls_emb = encoder(**tok).last_hidden_state[:, 0, :]
        # Gọi model (Unpack đúng 3 giá trị: score, masked_aij, raw_aij)
        final_score, m_aij_tensor, r_aij_tensor = model(cls_emb.unsqueeze(0), 
                                                        torch.tensor(m_ij.flatten(), device=device).unsqueeze(0))
        
        a_raw = r_aij_tensor.squeeze(0).cpu().numpy().reshape(10, 5)
        a_masked = m_aij_tensor.squeeze(0).cpu().numpy().reshape(10, 5)
        score = final_score.item()

    return cv_texts, jd_texts, a_raw, a_masked, score

# ═══════════════════════════════════════════════════════════════════════════
# 3. VẼ BIỂU ĐỒ
# ═══════════════════════════════════════════════════════════════════════════
for row in sample_to_vis:
    cv_t, jd_t, a_raw, a_masked, score = compute_heatmap_data(row)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))
    yticks = [get_clean_label(v, t, is_cv=True)  for v, t in zip(CV_VIEWS, cv_t)]
    xticks = [get_clean_label(v, t, is_cv=False) for v, t in zip(JD_VIEWS, jd_t)]

    # Heatmap 1: Before Masking (Màu xanh - Kết quả BERT học được)
    sns.heatmap(a_raw, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_xticklabels(xticks, rotation=45, ha="right")
    ax1.set_yticklabels(yticks, rotation=0)
    ax1.set_title(f"1. Before Masking (BERT Alignment)\nFile: {row['cv']}", fontsize=14, fontweight='bold')

    # Heatmap 2: After Masking (Màu đỏ - Kết quả đã lọc nhiễu)
    sns.heatmap(a_masked, annot=True, fmt=".2f", cmap="OrRd", ax=ax2, cbar_kws={'shrink': 0.8})
    ax2.set_xticklabels(xticks, rotation=45, ha="right")
    ax2.set_yticklabels(yticks, rotation=0)
    ax2.set_title(f"2. After Masking (Final Logic)\nJD: {row['jd']}", fontsize=14, fontweight='bold')

    plt.suptitle(f"SAMPLE ID: {row['id']} | Ground Truth: {row['score']:.4f} | AI Predicted: {score*100:.2f}%", 
                 fontsize=18, y=1.02, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print("═"*110)